In [1]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

Current Project Root Directory set to: /Users/luca/Documents/GitHub/US_conflict_forecasting
Current Search Path set to: ['/Users/luca/Documents/GitHub/US_conflict_forecasting', '/Users/luca/Documents/GitHub/US_conflict_forecasting/2_Dataset/c_Merge_Cleaning', '/Users/luca/anaconda3/envs/ML_Lab/lib/python312.zip', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/lib-dynload', '', '/Users/luca/anaconda3/envs/ML_Lab/lib/python3.12/site-packages', '/Users/luca/Documents/GitHub/US_conflict_forecasting']


In [2]:
#Loading data (all on All ZCTA, 12 month structure)
ACLED_nb = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_all_ZCTA_nb.csv'))

#Actor Features 
ACLED_prev_year = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_conflict_previousyear_allZCTA.csv'))

#Subevent Features 
ACLED_conf_all = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_conflict_all_ZCTA.csv'))

#Previous month
ACLED_prev_month = pd.read_csv(os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets/ACLED_conflict_prevmonth_allZCTA.csv'))

In [14]:
#Ensure that datasets are all in the same order + add index
#Sorting all ZIPs in ascending order 
ACLED_nb = ACLED_nb.sort_values(by=['ZCTA', 'month'], ascending=[True, True])
ACLED_prev_year = ACLED_prev_year.sort_values(by=['ZCTA', 'month'], ascending=[True, True])
ACLED_conf_all = ACLED_conf_all.sort_values(by=['ZCTA', 'month'], ascending=[True, True])
ACLED_prev_month = ACLED_prev_month.sort_values(by=['ZCTA', 'Month'], ascending=[True, True])

#Resetting index 
ACLED_conf_all.reset_index(drop=True, inplace=True)
ACLED_nb.reset_index(drop=True, inplace=True)
ACLED_prev_year.reset_index(drop=True, inplace=True)
ACLED_prev_month.reset_index(drop=True, inplace=True)

#Renaming column in prev month dataset
ACLED_prev_month = ACLED_prev_month.rename(columns={'event_count_prev': 'event_count_prev_month'})

Index(['ZCTA', 'Month', 'event_count', 'event_count_prev_month'], dtype='object')


In [15]:
#Merging on index: Merge conf_all with nb
Conflict_Features = ACLED_conf_all.merge(ACLED_nb, left_index=True, right_index=True, how='left', suffixes=('_conf', '_nb'))

#Merging conflicts in prev year 
Conflict_Features = Conflict_Features.merge(ACLED_prev_year, left_index=True, right_index=True, how='left', suffixes=('', '_prev_year'))

#Merging conflicts in prev month
Conflict_Features = Conflict_Features.merge(ACLED_prev_month, left_index=True, right_index=True, how='left', suffixes=('', '_prev_month'))

In [16]:
#Checking all col names
print(Conflict_Features.columns)

Index(['ZCTA_conf', 'month_conf', 'admin2_conf', 'fatalities_conf', 'act_gov',
       'act_pol', 'act_nat', 'act_rel', 'act_edu', 'act_wor', 'act_iss',
       'act_rwm', 'act_oth', 'set_peaceful_prot', 'set_other', 'set_prot_int',
       'set_mob', 'set_violent_prot', 'set_loot_destr', 'unions',
       'environment', 'gun-violence', 'healthcare', 'event_count_conf',
       'ZCTA_nb', 'month_nb', 'admin2_nb', 'fatalities_nb', 'event_count_nb',
       'neighbor_conflicts', 'ZCTA', 'month', 'admin2',
       'event_count_prev_year', 'ZCTA_prev_month', 'Month', 'event_count',
       'event_count_prev_month'],
      dtype='object')


In [17]:
#Dropping duplicate cols
Conflict_Features = Conflict_Features.drop(['ZCTA_nb', 'month_nb', 'admin2_nb', 'fatalities_nb', 'event_count_nb','ZCTA', 'month', 'admin2', 'ZCTA_prev_month', 'Month', 'event_count'], 
                                           axis=1)

In [18]:
Conflict_Features = Conflict_Features.rename(columns={'ZCTA_conf': 'ZCTA', 'month_conf': 'month', 'admin2_conf': 'admin2', 'event_count_conf': 'event_count', 'fatalities_conf': 'fatalities'})

In [19]:
print(Conflict_Features.columns.tolist())

['ZCTA', 'month', 'admin2', 'fatalities', 'act_gov', 'act_pol', 'act_nat', 'act_rel', 'act_edu', 'act_wor', 'act_iss', 'act_rwm', 'act_oth', 'set_peaceful_prot', 'set_other', 'set_prot_int', 'set_mob', 'set_violent_prot', 'set_loot_destr', 'unions', 'environment', 'gun-violence', 'healthcare', 'event_count', 'neighbor_conflicts', 'event_count_prev_year', 'event_count_prev_month']


In [20]:
#save as csv 
output_directory_1 = os.path.join(project_root_dir, '2_Dataset/b_Partial_Datasets')
output_filename_1 = "All_Conflict_Features.csv"
output_path_1 = os.path.join(output_directory_1, output_filename_1)

Conflict_Features.to_csv(output_path_1, index=False)